In [1]:
# ── Lightweight Model for Deployment ──────────────────────
# Uses TF-IDF + Logistic Regression instead of BERT
# RAM usage: ~50MB vs 512MB for DistilBERT

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
import joblib
import os

# Load cleaned data
df_train = pd.read_csv('../data/train_cleaned.csv')
df_test  = pd.read_csv('../data/test_cleaned.csv')

print("Train:", df_train.shape)
print("Test:", df_test.shape)
print("Classes:", df_train['status'].unique())

Train: (40012, 7)
Test: (992, 4)
Classes: ['Anxiety' 'Normal' 'Depression' 'Suicidal']


In [2]:
# Map labels to binary
label_map = {
    'Normal':     0,
    'Anxiety':    1,
    'Depression': 1,
    'Suicidal':   1,
}

df_train['label'] = df_train['status'].map(label_map)
df_test['label']  = df_test['status'].map(label_map)

df_train = df_train.dropna(subset=['label'])
df_test  = df_test.dropna(subset=['label'])

print("Train distribution:")
print(df_train['label'].value_counts())

Train distribution:
label
1    24021
0    15991
Name: count, dtype: int64


In [6]:
# Replace NaN with empty string
df_train['clean_text'] = df_train['clean_text'].fillna('')

# Build Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
    )),
])

# Train
print("Training...")
pipeline.fit(df_train['clean_text'], df_train['label'])
print("Done!")

Training...
Done!


In [7]:
# Evaluate
preds = pipeline.predict(df_test['clean_text'])
f1    = f1_score(df_test['label'], preds, average='weighted')

print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(
    df_test['label'], preds,
    target_names=['Normal', 'At-risk']
))

F1 Score: 0.8654

Classification Report:
              precision    recall  f1-score   support

      Normal       0.75      0.70      0.72       248
     At-risk       0.90      0.92      0.91       744

    accuracy                           0.87       992
   macro avg       0.83      0.81      0.82       992
weighted avg       0.86      0.87      0.87       992



In [8]:
# Save model
os.makedirs('../model/lightweight', exist_ok=True)
joblib.dump(pipeline, '../model/lightweight/model.joblib')
print("✅ Model saved to ../model/lightweight/model.joblib")

✅ Model saved to ../model/lightweight/model.joblib
